# Case Study: Fall of the Berlin Wall — From Minimal Input to Full HBI Output

This notebook illustrates the **full pipeline**: we start with minimal input (Year, Event Name, Continent), document the enrichment steps (each notebook and what it adds), then load the **result** of that pipeline and show segment distribution and dimension charts.

**Run from repository root** after `pip install -r requirements.txt`. See [berlin_wall_risk_register.md](berlin_wall_risk_register.md) for the full narrative and pipeline table.

## 1. Minimal starting point

The initial data had only **Year**, **Event Name**, and **Continent**. This is the input to the pipeline.

In [ ]:
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd()
CASE_STUDIES = REPO_ROOT / "case_studies"
SAMPLE_DATA = REPO_ROOT / "sample_data"

# Minimal input (Year, Event Name, Continent only)
df_minimal = pd.read_csv(CASE_STUDIES / "berlin_wall_minimal.csv")
df_minimal

## 2. Pipeline: notebooks run in order

To go from minimal input to full HBI output, run these notebooks **in this order**:

| Step | Notebook | Adds |
|------|----------|------|
| 1 | [Event_Description_notebook.ipynb](../notebooks/Data_Enrichment/Event_Description_notebook.ipynb) | **Event Description** |
| 2 | [Country_Prediction_notebook.ipynb](../notebooks/Data_Enrichment/Country_Prediction_notebook.ipynb) | **Country** |
| 3 | [Data_Enrichment_notebook.ipynb](../notebooks/Data_Enrichment/Data_Enrichment_notebook.ipynb) | **Event Type**, **Trigger**, **M, S, I, D, R** |
| 4 | [Data_Preprocessing.ipynb](../notebooks/Feature_Engineering/Data_Preprocessing.ipynb) | **Label columns** (Magnitude_label, Spread_label, etc.) |
| 5 | [Segment_Matrix.ipynb](../notebooks/Feature_Engineering/Segment_Matrix.ipynb) | **Mode**, **Segment** |

Below we load the **result** of this pipeline (pre-built for the Berlin Wall event).

## 3. Full output (result of pipeline)

Load the full gold-layer row from `example_berlin_wall.csv`.

In [ ]:
import pandas as pd
from pathlib import Path

# Assume run from repo root (see notebooks/README.md)
REPO_ROOT = Path.cwd()
CASE_STUDIES = REPO_ROOT / "case_studies"
SAMPLE_DATA = REPO_ROOT / "sample_data"

berlin_path = CASE_STUDIES / "example_berlin_wall.csv"
df_berlin = pd.read_csv(berlin_path)
df_berlin

## 4. Segment verification

Apply the canonical segment rules (from [docs/HBI_index_and_segment_matrix.md](../docs/HBI_index_and_segment_matrix.md)) to verify Mode and Segment.

In [ ]:
# Canonical segment logic (from docs/HBI_index_and_segment_matrix.md)
# Mode: S 1-3 -> Co-Present, S 4-10 -> Diffusive
def infer_mode(S):
    return "Diffusive" if S >= 4 else "Co-Present"

def classify_segment(M, S, I, D, mode):
    def is_low(val): return 1 <= val <= 3
    def is_medium(val): return 4 <= val <= 6
    def is_high(val): return 7 <= val <= 10
    def is_medium_or_above(val): return val >= 4
    if is_high(I) and is_high(D): return "S4 - Persistent Friction"
    if is_high(I) and not is_high(D): return "S2 - Emotion-Driven"
    if is_medium_or_above(M) and is_medium_or_above(S) and is_low(D): return "S3 - Volatile Expansion"
    if is_medium_or_above(M) and is_medium_or_above(S) and is_medium_or_above(D) and is_medium(I): return "S1 - Aligned Expansion"
    if is_low(I) and mode == "Diffusive": return "S5 - Low Energy Diffusion"
    return "Unclassified"

def assign_segment(row):
    M, S, I, D = row["Magnitude(M)"], row["Spread(S)"], row["Intensity(I)"], row["Duration(D)"]
    mode = infer_mode(S)
    return classify_segment(M, S, I, D, mode)

# Apply to Berlin Wall (already has Segment; we verify)
df_berlin["Mode_inferred"] = df_berlin["Spread(S)"].apply(infer_mode)
df_berlin["Segment_check"] = df_berlin.apply(assign_segment, axis=1)
df_berlin[["Event Name", "Year", "Magnitude(M)", "Spread(S)", "Intensity(I)", "Duration(D)", "Mode_inferred", "Segment", "Segment_check"]].to_string()

In [ ]:
# Combine with sample_events for a small portfolio and segment distribution
sample_path = SAMPLE_DATA / "sample_events.csv"
if sample_path.exists():
    df_sample = pd.read_csv(sample_path)
    if "Segment" not in df_sample.columns:
        df_sample["Mode_inferred"] = df_sample["Spread(S)"].apply(infer_mode)
        df_sample["Segment"] = df_sample.apply(assign_segment, axis=1)
    df_all = pd.concat([df_berlin, df_sample], ignore_index=True)
else:
    df_all = df_berlin.copy()

print("Segment distribution:")
print(df_all["Segment"].value_counts(dropna=False))

## 5. Segment distribution and dimension charts

Combine with sample events (if available) for segment distribution; show Berlin Wall dimension scores.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1. Segment distribution (bar)
counts = df_all["Segment"].value_counts()
axes[0].bar(counts.index, counts.values, color="steelblue", edgecolor="black")
axes[0].set_title("Segment distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# 2. Berlin Wall dimensions (horizontal bar)
dims = ["Magnitude(M)", "Spread(S)", "Intensity(I)", "Duration(D)", "Outcome/Impact(R)"]
vals = df_berlin[dims].iloc[0].values
axes[1].barh(dims, vals, color="coral", edgecolor="black")
axes[1].set_title("Berlin Wall (1989) — Dimension scores")
axes[1].set_xlim(0, 10)

plt.tight_layout()
plt.show()